In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
df = pd.read_csv("./data/auto-mpg.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())

In [ ]:

df["horsepower"] = df["horsepower"].replace("?", np.nan)
df["horsepower"] = df["horsepower"].astype(float)

print("Missing values in horsepower:", df["horsepower"].isna().sum())

In [ ]:
mean_hp = df["horsepower"].mean()
df["horsepower"].fillna(mean_hp, inplace=True)

print(f"Filled missing values with mean: {mean_hp:.2f}")
print("Missing values after filling:", df["horsepower"].isna().sum())

In [ ]:
X = df[["horsepower"]]
y = df["mpg"]

print("Feature (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nTarget statistics:")
print(y.describe())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
def polynomial_model(degree):

    poly = PolynomialFeatures(degree=degree)
    
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    y_pred = model.predict(X_test_poly)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    return model, poly, mse, rmse, r2



In [ ]:
results = {}

degrees = [1, 2, 3, 4, 5]

print("Training polynomial models...\n")
for d in degrees:
    model, poly, mse, rmse, r2 = polynomial_model(d)
    results[d] = {"MSE": mse, "RMSE": rmse, "R²": r2}
    print(f"Degree {d}: RMSE={rmse:.4f}, R²={r2:.4f}")

results_df = pd.DataFrame(results).T
print("\n" + "="*60)
print("POLYNOMIAL REGRESSION RESULTS")
print("="*60)
print(results_df)
print("="*60)

### 2.9 Model Comparison Table

| **Degree** | **Behavior**                    | **Typical Outcome**        |
|------------|---------------------------------|----------------------------|
| 1          | Linear fit                      | Underfitting               |
| 2          | Quadratic curve                 | Good bias-variance balance |
| 3          | Cubic curve                     | Better fit                 |
| 4          | Higher-order polynomial         | Overfitting risk           |
| 5          | Very complex curve              | High overfitting risk      |

In [ ]:

X_range = np.linspace(X.min().values[0], X.max().values[0], 300).reshape(-1, 1)
X_range_scaled = scaler.transform(X_range)

plt.figure(figsize=(12, 7))
plt.scatter(X, y, alpha=0.4, s=30, c='gray', label="Actual Data", edgecolors='k', linewidths=0.5)

colors = ['blue', 'green', 'orange', 'red', 'purple']
for i, d in enumerate([1, 2, 3, 4, 5]):
    poly = PolynomialFeatures(d)
    X_poly = poly.fit_transform(X_train_scaled)
    model = LinearRegression()
    model.fit(X_poly, y_train)
    
    y_curve = model.predict(poly.transform(X_range_scaled))
    plt.plot(X_range, y_curve, label=f"Degree {d}", linewidth=2.5, color=colors[i])

plt.xlabel("Horsepower", fontsize=12)
plt.ylabel("MPG (Miles Per Gallon)", fontsize=12)
plt.title("Polynomial Regression: Effect of Degree on Model Fit", fontsize=14, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()